# Practical Sumcheck in Sagemath

**Reference:** https://blog.yaudit.dev/sumcheck

**Table of content:**

- Sagemath/Algebra refresher
- Multivariate and multilinear polynomials
- Boolean hypercube
- Multilinear extensions
- Equality function
- Sumcheck

## Sagemath syntax that we need

In [1]:
# Define F to be the Galois Field GF(17), also called Finite Field. And this is just Z/17Z since 17 is a prime.
F = GF(17)
print("F: ", F)

# 1. Creates a Ring: It builds the mathematical structure of polynomials.
# 2. Names the Ring: It assigns that structure to the Python variable R.
# 3. Injects Variables: It automatically defines x, y, and z as global variables that you can use immediately.
R.<x, y, z> = PolynomialRing(F)
print("R: ", R)
# Or (eqivalent syntax):
T.<x, y, z> = F[]
print("T: ", T)

F:  Finite Field of size 17
R:  Multivariate Polynomial Ring in x, y, z over Finite Field of size 17
T:  Multivariate Polynomial Ring in x, y, z over Finite Field of size 17


## Algebra background

**ATTENTION: To make things simple, we only consider Galois field with prime order -> $GF(p)$! That is equivalent to $\mathbb{Z}/p\mathbb{Z}$. (GF can be $GF(p)$ or $GF(p^k)$, we only consider the former.)**

### (optional) 0 - Group, ring, field, and polynomial ring

#### Group

A **group** is a pair $(G, \cdot)$ consisting of a set $G$ and a binary operation
$$
\cdot : G \times G \to G
$$
such that:

1. **Associativity**  
   For all $a,b,c \in G$,
   $$
   (a \cdot b) \cdot c = a \cdot (b \cdot c).
   $$

2. **Identity element**  
   There exists an element $e \in G$ (called the identity) such that for all $a \in G$,
   $$
   e \cdot a = a \cdot e = a.
   $$

3. **Inverses**  
   For every $a \in G$, there exists an element $a^{-1} \in G$ such that
   $$
   a \cdot a^{-1} = a^{-1} \cdot a = e.
   $$

If, in addition,

4. **Commutativity**  
   For all $a,b \in G$,
   $$
   a \cdot b = b \cdot a,
   $$

then $(G,\cdot)$ is called an **abelian group**.

---

#### Ring

A **ring** is a triple $(R, +, \cdot)$ where $R$ is a set equipped with two binary operations
$$
+ : R \times R \to R,\quad \cdot : R \times R \to R
$$
such that:

1. $(R, +)$ is an abelian group:
   - (Associativity) For all $a,b,c \in R$,
     $$
     (a + b) + c = a + (b + c).
     $$
   - (Identity) There exists $0 \in R$ such that for all $a \in R$,
     $$
     a + 0 = 0 + a = a.
     $$
   - (Inverses) For each $a \in R$ there exists $-a \in R$ such that
     $$
     a + (-a) = (-a) + a = 0.
     $$
   - (Commutativity) For all $a,b \in R$,
     $$
     a + b = b + a.
     $$

2. $(R, \cdot)$ is a semigroup:
   - (Associativity) For all $a,b,c \in R$,
     $$
     (a \cdot b) \cdot c = a \cdot (b \cdot c).
     $$

3. **Distributive laws** hold:
   - For all $a,b,c \in R$,
     $$
     a \cdot (b + c) = a \cdot b + a \cdot c,
     $$
     $$
     (a + b) \cdot c = a \cdot c + b \cdot c.
     $$

A ring is called a **unital ring** (or a “ring with identity”) if:

4. **Multiplicative identity**  
   There exists $1 \in R$ such that, for all $a \in R$,
   $$
   1 \cdot a = a \cdot 1 = a.
   $$

A ring is called **commutative** if, for all $a,b \in R$,
$$
a \cdot b = b \cdot a.
$$

(When someone says “ring” without qualification, they often mean a *commutative unital* ring, but conventions vary.)

---

#### Field

A **field** is a commutative unital ring $(F, +, \cdot)$ with $1 \neq 0$ in which every nonzero element is multiplicatively invertible.

More explicitly, $(F,+,\cdot)$ is a field if:

1. $(F, +)$ is an abelian group (as above), with additive identity $0$.

2. $(F \setminus \{0\}, \cdot)$ is an abelian group:
   - (Associativity) For all $a,b,c \in F \setminus \{0\}$,
     $$
     (a \cdot b) \cdot c = a \cdot (b \cdot c).
     $$
   - (Identity) There exists an element $1 \in F \setminus \{0\}$ such that for all $a \in F \setminus \{0\}$,
     $$
     1 \cdot a = a \cdot 1 = a.
     $$
   - (Inverses) For each $a \in F \setminus \{0\}$ there exists $a^{-1} \in F \setminus \{0\}$ such that
     $$
     a \cdot a^{-1} = a^{-1} \cdot a = 1.
     $$
   - (Commutativity) For all $a,b \in F \setminus \{0\}$,
     $$
     a \cdot b = b \cdot a.
     $$

3. **Distributive laws** (same as for rings) hold for all $a,b,c \in F$:
   $$
   a \cdot (b + c) = a \cdot b + a \cdot c,\quad
   (a + b) \cdot c = a \cdot c + b \cdot c.
   $$

Equivalently: a field is a commutative ring in which $0 \neq 1$ and every nonzero element is a unit.

#### Polynomial ring

Let $R$ be a commutative ring (with identity). The **polynomial ring** in one indeterminate $x$ over $R$ is denoted $R[x]$.

**As a set**, $R[x]$ consists of all formal expressions
$$
f(x) = a_0 + a_1 x + a_2 x^2 + \cdots + a_n x^n
$$
with:

- $n \in \mathbb{N}$ (possibly $n = 0$),
- coefficients $a_i \in R$,
- and **only finitely many** nonzero coefficients (so each polynomial has finite degree).

Two such expressions are equal if and only if their corresponding coefficients are equal.

**Operations** on $R[x]$:

1. **Addition**: If
   $$
   f(x) = \sum_{i=0}^m a_i x^i,\quad
   g(x) = \sum_{i=0}^n b_i x^i,
   $$
   then
   $$
   f(x) + g(x) = \sum_{i=0}^{\max(m,n)} (a_i + b_i) x^i,
   $$
   where missing coefficients are interpreted as $0$.

2. **Multiplication**: Defined by the Cauchy product of coefficients:
   $$
   f(x) \cdot g(x)
   = \left( \sum_{i=0}^m a_i x^i \right)
     \left( \sum_{j=0}^n b_j x^j \right)
   = \sum_{k=0}^{m+n} c_k x^k,
   $$
   where
   $$
   c_k = \sum_{i+j = k} a_i b_j.
   $$

With these operations, $R[x]$ is a commutative ring with identity; the constant polynomial $1$ acts as the multiplicative identity, and the constant polynomial $0$ as the additive identity.

For several indeterminates $x_1, \dots, x_n$, one defines $R[x_1, \dots, x_n]$ analogously as the ring of polynomials in $x_1,\dots,x_n$ with coefficients in $R$.

### (optional) 1 - Examples of groups/rings/fields

#### Groups

##### Additive groups

- $(\mathbb{Z}, +)$ – integers under addition  
  - Infinite, abelian, generated by $1$.

- $(\mathbb{Z}/n\mathbb{Z}, +)$ – integers modulo $n$ under addition  
  - Finite, abelian, of order $n$, cyclic.

##### Multiplicative groups

- $(\mathbb{Q}^\times, \cdot)$, $(\mathbb{R}^\times, \cdot)$, $(\mathbb{C}^\times, \cdot)$  
  - Nonzero rationals/reals/complex numbers under multiplication.

- $(\mathbb{Z}/n\mathbb{Z})^\times$ or $(\mathbb{Z}/n\mathbb{Z})^*$ – units modulo $n$  
  - Elements in $\{1,\dots,n-1\}$ that are coprime to $n$, with multiplication mod $n$.

#### Rings

##### Familiar number rings

- $(\mathbb{Z}, +, \cdot)$ – integers  
  - Commutative ring with $1$, not a field.

- $(\mathbb{Z}/n\mathbb{Z}, +, \cdot)$ – integers mod $n$  
  - Commutative ring with $1$; a field only when $n$ is prime.

##### Polynomial rings

- $\mathbb{Z}[x]$, $\mathbb{Q}[x]$, $\mathbb{R}[x]$, $\mathbb{F}_p[x]$  
  - Polynomials in one variable with coefficients in the given ring/field.

- $\mathbb{R}[x,y]$, $\mathbb{C}[x_1,\dots,x_n]$  
  - Polynomials in several variables.

##### Matrix rings

- $M_n(R)$ – all $n \times n$ matrices over a ring $R$  
  - Ring under matrix addition and multiplication.
  - For $R$ commutative, $M_n(R)$ is usually noncommutative for $n \ge 2$.

Common special case:
- $M_n(\mathbb{R})$, $M_n(\mathbb{C})$.

#### Fields

##### Classical infinite fields

- $\mathbb{Q}$ – rational numbers  
- $\mathbb{R}$ – real numbers  
- $\mathbb{C}$ – complex numbers  

All are fields under usual addition and multiplication.

##### Finite fields (Galois fields)

- $\mathbb{F}_p = \mathbb{Z}/p\mathbb{Z}$, where $p$ is prime  
  - Field of $p$ elements, e.g. $\mathbb{F}_2, \mathbb{F}_3, \mathbb{F}_5$.

- $\mathbb{F}_{p^n}$ – any finite field with $p^n$ elements  
  - Often built as $\mathbb{F}_p[x]/(f(x))$ with $f$ irreducible of degree $n$.

### 2 - $\mathbb{Z}/n\mathbb{Z}$ under adddition

#### Intuitive understanding

Think of $\mathbb{Z}/n\mathbb{Z}$ as clock arithmetic. For example, $\mathbb{Z}/12\mathbb{Z}$ represents a real-world clock:

- The algebraic structure holds $\{0, 1, 2, 3, ..., 11\}$ -> from $0$ to $n - 1$.
- $11 + 1 = 12 = 0$, it wraps around.

#### Defining $\mathbb{Z}/n\mathbb{Z}$ based on equivalence relation

The equivalence relation that defines $\mathbb{Z}/n\mathbb{Z}$ is:

$$
a \sim b \quad \Longleftrightarrow \quad n \text{ divides } (a - b),
$$

or equivalently,

$$
a \sim b \quad \Longleftrightarrow \quad a \equiv b \pmod{n}.
$$

Each equivalence class under this relation is

$$
[a] = \{\, a + kn : k \in \mathbb{Z} \,\},
$$

and $\mathbb{Z}/n\mathbb{Z}$ is the set of all such equivalence classes:

$$
\mathbb{Z}/n\mathbb{Z} = \{[0], [1], \dots, [n-1]\}.
$$

#### Appendix: Proving congruence modulo n is an equivalence relation

To be an equivalence relation, the relation must satisfy three properties: reflexivity, symmetry, and transitivity:

- **Reflexivity:** $a \equiv a \mod n$
- **Symmetry:** if $a \equiv b \mod n$ then $b \equiv a \mod n$
- **Transitivity:** if $a \equiv b \mod n$ and $b \equiv c \mod n$ then $a \equiv c \mod n$

### 3 - $(\mathbb{Z}/n\mathbb{Z})^*$ under multiplication

$(\mathbb{Z}/n\mathbb{Z})^*$ is the part of $\mathbb{Z}/n\mathbb{Z}$ where **multiplication behaves like with ordinary nonzero numbers**: you can divide (i.e., every element has a multiplicative inverse).

---

#### $\mathbb{Z}/n\mathbb{Z}$ under multiplication isn't a group

Not every nonzero element in $\mathbb{Z}/n\mathbb{Z}$ has an inverse under multiplication.

Example in $\mathbb{Z}/8\mathbb{Z}$:

- $1$ has an inverse: $1\cdot1 \equiv 1\pmod{8}$.
- $3$ has an inverse: $3\cdot3 = 9 \equiv 1\pmod{8}$.
- $5$ has an inverse: $5\cdot5 = 25 \equiv 1\pmod{8}$.
- $7$ has an inverse: $7\cdot7 = 49 \equiv 1\pmod{8}$.

But:

- $2\cdot k$ is always even, so it's never $1 \pmod{8}$. So $2$ has no inverse.
- Same for $4,6$.

So in $\mathbb{Z}/8\mathbb{Z}$, some nonzero elements are invertible and some are non-invertible.

---

#### What is $(\mathbb{Z}/n\mathbb{Z})^*$?

$$
(\mathbb{Z}/n\mathbb{Z})^* = \{\text{all elements of } \mathbb{Z}/n\mathbb{Z} \text{ that have a multiplicative inverse}\}.
$$

Formally:

$$
(\mathbb{Z}/n\mathbb{Z})^* = \{[a] \in \mathbb{Z}/n\mathbb{Z} : \exists [b] \text{ with } [a][b] = [1]\}.
$$

---

#### Which elements are in $(\mathbb{Z}/n\mathbb{Z})^*$?

Key fact:

> $[a]$ is invertible mod $n$ iff $\gcd(a,n) = 1$.

So:

- Take all numbers between $0$ and $n-1$.
- Keep only those that are coprime to $n$ (share no common prime factor with $n$, except 1).
- Their residue classes form $(\mathbb{Z}/n\mathbb{Z})^*$.

Examples:

- For $n=8$: numbers $1,3,5,7$ are coprime to 8.  
  So
  $$
  (\mathbb{Z}/8\mathbb{Z})^* = \{1,3,5,7\}.
  $$

- For $n=10$: numbers coprime to 10 in $\{0,\dots,9\}$ are $1,3,7,9$.  
  So
  $$
  (\mathbb{Z}/10\mathbb{Z})^* = \{1,3,7,9\}.
  $$

- If $p$ is prime, then every nonzero element $1,2,\dots,p-1$ is coprime to $p$.  
  Thus
  $$
  (\mathbb{Z}/p\mathbb{Z})^* = \{1,2,\dots,p-1\},
  $$
  i.e., everything except $0$.

### 4 - $\mathbb{Z}/p\mathbb{Z}$

When restricting the $n$ in $\mathbb{Z}/p\mathbb{Z}$ to be a prime $p$, the algebraic structure is denoted as $\mathbb{Z}/p\mathbb{Z}$.

$\mathbb{Z}/p\mathbb{Z}$ is a field.

**Proof sketch:** $p$ is a prime. Pick $a$ s.t. $1 \leq a < p$, $gcd(a, p) = 1$, then by Bezout's identity $ax + py = 1$ for some integer $x$ and $y$. Take mod $p$ on both sides we get $ax = 1 \mod p$, so $x$ is the inverse of a in $\mathbb{Z}/p\mathbb{Z}$.

Since finite field is defined to be a field with finite elements, $\mathbb{Z}/p\mathbb{Z}$ is finite field.

## Multivariate and multilinear polynomials

A **multivariate polynomial** is a polynomial with more than one variable. For example:

$$f(x, y) = 3x^3y^4 + 2xy^2 + 5y + 7$$

Degree of $x$ is 3 and degree of $y$ is 4 (highest degree) and degree for the entire polynomial is 3 + 4 = 7 from the term with highest combined degree ($3x^3y^4$ in this example).

A **multilinear polynomial** is a special type of multivariate polynomial where each variable appears with at most a degree of 1 (i.e., no exponents greater than 1). For example:

$$f(x, y, z) = x * y * z + x * z + z + 5$$

The degree of the polynomial is at max the number of variables ($x * y * z$ in this example).

In [4]:
F = GF(17)
R.<x, y, z> = PolynomialRing(F)

P = 6*x^3*z^3*y - 6*z^2 - x - 2*y + 6 # P is defined to be an element in R implicitly
assert(P.degree() == 7)

P = R.random_element(degree=5)
print("deg 5 poly", P)

print("\nP * z = ", P * z)

print("----------")

P = R.random_element(degree=2)
print("random P:", P)
print("P(1,2,3):", P(1,2,3))

print("----------")

print("Other example, with multilinear terms")
print(5*x*y + 7*y + x*y*z + 7)

deg 5 poly -x^2*z^3 + 8*x^2*z^2 - 8*y*z^3 - 5*x*y

P * z =  -x^2*z^4 + 8*x^2*z^3 - 8*y*z^4 - 5*x*y*z
----------
random P: 7*y^2 + 8*x*z - 4*y*z - 4*z^2 + 6*x
P(1,2,3): 15
----------
Other example, with multilinear terms
x*y*z + 5*x*y + 7*y + 7


## Boolean hypercube

An **n-dimensional hypercube** consists of all possible tuples of length $n$ where each element comes from a fixed set. For example, if we pick the elements $\{a, b\}$, the 3D hypercube is:

```
(a, a, a), (a, a, b), (a, b, a), (a, b, b), (b, a, a), (b, a, b), (b, b, a), (b, b, b)
```

Set a = 0 and b = 1 and we get **boolean hypercube**:

```
(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)
```

**n-dimensional boolean hypercube** is just a list of all numbers in binary representation from $0$ to $2^n - 1$:

```
# Decimal representation
[0, 1, 2, 3, 4, 5, 6, 7]

# Binary strings representation
['000', '001', '010', '011', '100', '101', '110', '111']

# Tuples representation
[[0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 1], [1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]]
```


## Evaluating the polynomial over the hypercube

Evaluating $f(x, y, z) = x * y + z + 2$ over 3d hypercube `[[0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 1], [1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]]`:

```
f(0, 0, 0) + f(0, 0, 1)
+ f(0, 1, 0) + f(0, 1, 1)
+ f(1, 0, 0) + f(1, 0, 1)
+ f(1, 1, 0) + f(1, 1, 1)
```


In [15]:
import itertools

# Use itertools instead of list comprehension since later can't do variable dimension

dimension = 3
cube = list(itertools.product([0, 1], repeat=dimension))

F = GF(17)
R.<a, b, c> = F[] # Polynomials with coefficients in F

# this one is multilinear, but it doesn't change anything
P = 6*a*b*c + 5*a*b + 4*a*c + 3*a + 2*b*c + b + 7

# we can check the degrees of each variable
assert(P.degrees() == (1,1,1))
print("P:", P)

# we can evaluate the polynomial over the hypercube
S = sum([P(i) for i in cube])
print("Sum of the polynomial over the hypercube:", S)

P: 6*a*b*c + 5*a*b + 4*a*c + 2*b*c + 3*a + b + 7
Sum of the polynomial over the hypercube: 15


## Multilinear extensions (MLE)

**Multilinear extension** is a multilinear polynomial that accepts more than boolean inputs. To get the MLE of a function, we interpolate (langrange interpolation) a multilinear polynomial from $\{0, 1\}^n$ inputs. This interpolated polynomial is initially only defined over $\{0,1\}$ inputs. However, we can evaluate this polynomial at non-boolean values (like $2$, $3.5$, or -1), effectively "extending" it beyond the boolean hypercube.

> **Lagrange interpolation:** https://youtu.be/bzp_q7NDdd4?si=b88ACQg_4u-Z0IZq. This topic was discussed in depth in "QAP and Pairing" talk.

> If we start with a function $f(x_1, ..., x_n)$ defined over the boolean hypercube, we can find many multivariate polynomials over $\mathbb{F}$ that agree with $f$ on boolean inputs. However, there’s exactly one polynomial that both agrees with $f$ on the boolean hypercube and is multilinear (has degree at most $1$ in each variable): this unique multilinear polynomial is what we call the multilinear extension of $f$.

Why "we can find multivariate polynomials over $\mathbb{F}$ that agree with $f$ on boolean inputs"?

- $0^2 = 0$
- $1^2 = 1$

Because $x^2 = x$ for these specific inputs, the polynomials $P(x) = x$ and $Q(x) = x^2$ look identical. Since we can add any power ($x^2, x^3, x^{100}$) without changing the result on boolean inputs, there are infinite polynomials to choose from.

But when we restrict the multivariable polynomial to be multilinear, the result is unique. For multilinear interpolation, we work with boolean inputs ($0$ and $1$), so we use $x$ and $1 - x$ as "selectors" to construct basis polynomials.

- $x$ evaluates to $1$ when $x = 1$ and $0$ when $x = 0$.
- $(1 - x)$ evaluates to $1$ when $x = 0$ and $0$ when $x = 1$.

Let's look at a 2-variable example, $f(x_1, x_2)$. There are 4 corners on the boolean square: $(0,0), (0,1), (1,0), (1,1)$. We construct the polynomial by summing up terms for each corner:

```
f̃(x₁, x₂) =
f(0,0) · (1 - x₁)(1 - x₂)    // This term is active only when x₁ = 0, x₂ = 0
+ f(0,1) · (1 - x₁)(x₂)      // This term is active only when x₁ = 0, x₂ = 1
+ f(1,0) · (x₁)(1 - x₂)      // This term is active only when x₁ = 1, x₂ = 0
+ f(1,1) · (x₁)(x₂)          // This term is active only when x₁ = 1, x₂ = 1
```

The MLE of a function is often written with `~`. So the MLE of $f(x)$ will be written as $\tilde{f}(x)$.

**Note**: You will notice the algorithm is similar but not the same as traditional lagrange interpolation. This is expected and we can YOLO it and just think it is the same as lagrange.

**Another note:** After interpolating a multilinear poly, why we can evaluate it outside boolean hypercube? Because the interpolation produces an algebraic structure (a polynomial) defined over the entire field, not just a lookup table defined on the hypercube.


In [14]:
F = GF(101)
R.<a,b> = F[]

def multilinear_lagrange_interpolation(points, values):
    def basis_polynomial(point):
        result = a if point[0] == 1 else (1 - a)
        result *= b if point[1] == 1 else (1 - b)
        return result

    interpolation_polynomial = R(0) # 0 casted to polynomial ring

    for i in range(len(points)):
        interpolation_polynomial += F(values[i]) * basis_polynomial(points[i])

    return interpolation_polynomial

points = [(0, 0), (0, 1), (1, 0), (1, 1)]

P = multilinear_lagrange_interpolation(points, values)
print("P: ", P)

P:  2*a*b + 4*a + 2*b + 3


## Equality function

**Equality function** is a function takes two inputs $x$ and $y$, each in $\{0, 1\}^n$, and returns:

- $eq(x,y) = 1$, if $x = y$
- $eq(x,y) = 0$, if $x \neq y$

Its multilinear extension over a field $\mathbb{F}$ is given by:

$$
eq(x, y) = \prod_{i=0}^{n-1} (x_i y_i + (1 - x_i)(1 - y_i))
$$

The formula says if each bit of $x$ match the corresponding bit of $y$ then return 1, else return 0. Take $\{0, 1\}^2$ as example, for each term inside the multiplication sign:

- $term_i = x_i y_i + (1 - x_i)(1 - y_i)$
- If $x_i = 1$ and $y_i = 1$: $1(1) + (0)(0) = 1 + 0 = \mathbf{1}$ (Match)
- If $x_i = 0$ and $y_i = 0$: $0(0) + (1)(1) = 0 + 1 = \mathbf{1}$ (Match)
- If $x_i = 1$ and $y_i = 0$: $1(0) + (0)(1) = 0 + 0 = \mathbf{0}$ (Mismatch)
- If $x_i = 0$ and $y_i = 1$: $0(1) + (1)(0) = 0 + 0 = \mathbf{0}$ (Mismatch)

After we multiply all the terms, if the final result is 1 then it means all the terms are in "Match" state.

In [11]:
F = GF(101)
R.<x1,x2,x3,y1,y2,y3> = F[]

cube = [(0,0,0),(0,0,1),(0,1,0),(0,1,1),(1,0,0),(1,0,1),(1,1,0),(1,1,1)]

def eq(y):
    beta = x1 * y[0] + (1 - x1) * (1 - y[0])
    beta *= x2 * y[1] + (1 - x2) * (1 - y[1])
    beta *= x3 * y[2] + (1 - x3) * (1 - y[2])
    return beta

def mle(values):
    return sum(values[i] * eq(c) for i,c in enumerate(cube))

a = [83,10,50,5,19,23,90,75]
P = mle(a)
print("P: ", P)

P:  -47*x1*x2*x3 + 3*x1*x2 - 24*x1*x3 + 28*x2*x3 + 37*x1 - 33*x2 + 28*x3 - 18


## Sumcheck

**Intuition behind Sumcheck: Recursively eliminate one variable with a random challenge until all variables are replaced by randoms, then check correctness relying on Schwartz-Zippel Lemma**

The protocol starts with a multivariate polynomial:

$$
P(x_1, x_2, ..., x_n)
$$

We then evaluate this polynomial over all points in the boolean hypercube:

$$
H = \sum_{v_1 \in \{0,1\}} \sum_{v_2 \in \{0,1\}} ... \sum_{v_n \in \{0,1\}} P(v_1, v_2, ..., v_n)
$$

The goal of is to allow a prover to convince a verifier that this sum is correct, without requiring the verifier to compute every evaluation directly.

We use a 2-variable polynomial $P(x_1, x_2)$ evaluated over the boolean hypercube as example, producing a result $H$:

$$
H = \sum_{x_1, x_2 \in \{0,1\}} P(x_1, x_2)
$$

Expanding this sum, we get:

$$
H = P(0,0) + P(0,1) + P(1,0) + P(1,1)
$$

The goal of the protocol is to gradually reduce the number of variables in $P$, one by one, while ensuring correctness at each step.

### Round 1

At each round, we eliminate one variable by constructing a new polynomial that depends on one less variable. To do this, we define a univariate polynomial that captures the contribution of one variable at a time. Specifically, we create a new polynomial $g_1(x)$ in which $x_1$ remains a variable, but $x_2$ is summed out:

$$
g_{1}(x_{1}) = P(x_{1}, 0) + P(x_{1}, 1)
$$

The total sum $H$ becomes the following under this construction:

$$
H = g_{1}(0) + g_{1}(1)
$$

$H$ is the public input that prover sent to verifier at the very beginning, therefore verifier already knew what $H$ is. In this step Prover also sends $g_1(x1)$ to verifier.

Verifier then computes $g_1(0)$ and $g_1(1)$ and checks if they sum to $H$:

$$
H \stackrel{?}{=} g_1(0) + g_1(1)
$$

Once the verifier is convinced that $g_{1}(x)$ is correct, they choose a random point $r_1$ and ask the prover to evaluate $P$ at $x_1 = r_1$, reducing the number of variables: $P(r_1, x_2)$. (This step is for detecting forged $g$ from prover -> verifier is going to issue such randomness challege in each round, and in the end verifier will get a $P$ evaluated at all the random points and check the correctness of it in the last round.)

### Round 2

Prover evaluates $P(r_1, x_2)$ given the random $r_1$ from verifier and sets a new poly $g_2(x_2)$ and sends $g_2(x_2)$ to verifier:

$$
g_2(x_2) = P(r_1, x_2)
$$

Verifier evaluates $g_2(0)$ and $g_2(1)$ then checks:

$$
g_2(0) + g_2(1) \stackrel{?}{=} g_1(r_1)
$$

### Final round

Again, verifier samples another random $r_2$ and let prover evaluates $P(r_1, r_2)$. This value is sent back to verifier and verifier checks:

$$
P(r_1, r_2) \stackrel{?}{=} g_2(r_2)
$$

The check above prevents malicious prover from forging fake $P$ or $g$ because of Schwartz-Zippel Lemma.

In [17]:
F = GF(101)

# Define a polynomial ring with two variables
R.<x1,x2> = F[]

# Define a univariate polynomial ring (for later reductions)
Rx.<x> = F[]

# Define a random multilinear polynomial
P = 1 + x2 + x1 + x1*x2 # P is implicitly an element of the polynomial ring R

# Compute the sum of P over the boolean hypercube
H = P(0, 0) + P(0, 1) + P(1, 0) + P(1, 1)
print(f"H = {H}")

# Prover constructs g1(x) = P(x,0) + P(x,1)
g1 = P(x1,0) + P(x1,1)

# Convert g1 to a univariate polynomial in x (just for clarity)
g1 = Rx(g1(x, 0))
print(f"g1 = {g1}")

# Verifier checks if g1 correctly sums to H
print(f"g1(0) + g1(1) = {g1(0) + g1(1)} (should equal {H})")
assert g1(0) + g1(1) == H

# Verifier sends a random challenge r1
r1 = 31

# Prover constructs g2(x) = P(r1, x)
g2 = P(r1, x2)

# Convert g2 to a univariate polynomial in x (for clarity)
g2 = Rx(g2(0, x))
print(f"g2 = {g2}")

# Verifier checks if g2 correctly sums to g1(r1)
print(f"g2(0) + g2(1) = {g2(0) + g2(1)} (should equal {g1(r1)})")
assert g2(0) + g2(1) == g1(r1)

r2 = 43
assert g2(r2) == P(r1, r2)

H = 9
g1 = 3*x + 3
g1(0) + g1(1) = 9 (should equal 9)
g2 = 32*x + 32
g2(0) + g2(1) = 96 (should equal 96)


## Implement Sumcheck and HyperPlonk on your own

https://sumcheck.zksecurity.xyz/